### Embeddings Techniques

In [3]:
# Use it in LangChain:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Single word or text
vector = embeddings.embed_query("king")
print(len(vector))   # vector dimension
print(vector[:50])    # first few values

# Many words/texts at once
# vectors = embeddings.embed_documents(["king", "queen", "apple"])

768
[-0.01529125, 0.054462865, -0.17045893, -0.017364277, -0.040041313, 0.040827382, -0.057207163, 0.05213511, -0.0039191837, -0.024853585, -0.058646098, 0.05283702, 0.03330843, 0.013433285, -0.0104314955, -0.0115436, -0.022540065, -0.0061636316, -0.0020085664, 0.037011277, -0.036095057, 0.019243961, -0.036014, 0.03785465, 0.15297365, -0.007382636, 0.000106397856, -0.002579102, -0.018203853, -0.008738017, 0.031077577, 0.04109324, -0.014870283, 0.017519385, -0.005623884, -0.041224558, 0.062066972, -0.03121452, 0.08643195, -0.06512889, 0.036467206, -0.013576255, -0.06918333, 0.020447008, 0.061129592, -0.05790521, 0.015225184, 0.08846481, -0.030227274, -0.010617265]


In [4]:
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

DB_DIR = "./chroma_db"        # folder where the vector DB is saved
COLLECTION = "speech"         # name of the collection inside the DB

# Embedding model (must match the one used when the DB was created)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

if os.path.exists(DB_DIR):
    # DB already exists: load it instead of re-embedding
    vector_store = Chroma(
        collection_name=COLLECTION,
        embedding_function=embeddings,
        persist_directory=DB_DIR,
    )
    print("Loaded existing vector DB")
else:
    # Load the file
    docs = TextLoader("speech.txt", encoding="utf-8").load()

    # Split into chunks (embedding models have input limits)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)
    print(f"{len(chunks)} chunks")

    # Embed the chunks and store them in the vector DB (saved to disk automatically)
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION,
        persist_directory=DB_DIR,
    )
    print("Created vector DB and stored embeddings")

# Quick check: search by meaning
results = vector_store.similarity_search("What is the main message of the speech?", k=3)
for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n{r.page_content}")

/var/folders/4d/q3cdgnnx1_z7vjryt97hsv7w0000gq/T/ipykernel_79351/4115537206.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


10 chunks
Created vector DB and stored embeddings

--- Result 1 ---
and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.

--- Result 2 ---
always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.

--- Result 3 ---
It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to 

In [5]:
print('*' * 12)

************


In [7]:
queries = [
    "making the world safe for democracy",
    "the US does not want conquest or compensation",
    "attitude toward the German people",
    "loyalty of German-Americans",
    "the cost and sacrifice of going to war",
]

for q in queries:
    doc, score = vector_store.similarity_search_with_score(q, k=1)[0]
    print(f"\nQ: {q}\n{score:.4f}  {doc.page_content[:120]}...")


Q: making the world safe for democracy
0.4121  The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. W...

Q: the US does not want conquest or compensation
0.8451  The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. W...

Q: attitude toward the German people
0.5953  We have borne with their present government through all these bitter months because of that friendship—exercising a pati...

Q: loyalty of German-Americans
0.5824  are in fact loyal to their neighbors and to the government in the hour of test. They are, most of them, as true and loya...

Q: the cost and sacrifice of going to war
0.8067  It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There...
